# 🤖 Explore Isaac Lab
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isaac-sim/IsaacLab/blob/develop/notebooks/explore.ipynb)

**Walk. Grasp. See. Bend.** Try a robot, watch its replay, then change one thing.

🐾 Locomotion → 🦾 Manipulation → 📷 Robot vision → 🪢 Cloth

Choose **Runtime → Change runtime type → GPU**, then run downward. Videos appear beneath each demo.


## 1 · Power on
Setup takes a few minutes; later runs reuse downloaded assets. Expand **Show log** if something fails.

<details><summary>What gets installed?</summary>

Isaac Lab 3.0's development branch, Newton, OV PhysX, OV RTX, and video support. The GPU's drivers must support the selected backend. Use a release tag for a workshop; the installed revision is shown below.

</details>


In [ ]:
# @title Prepare your robot playground
import contextlib
import html
import os
import re
import signal
import subprocess
import sys
import time
import uuid
from pathlib import Path

from IPython.display import HTML, Video, display

SESSION = Path("/content/isaaclab_notebook_logs")
SESSION.mkdir(parents=True, exist_ok=True)


def run(command, *, cwd=None, env=None, check=True, label="Working"):
    """Show progress, save full logs, and stop child processes when interrupted."""
    log = SESSION / f"{uuid.uuid4().hex}.log"
    status = display(HTML(""), display_id=True)
    started = time.monotonic()
    with log.open("w") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=stream,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        try:
            while process.poll() is None:
                with log.open("rb") as reader:
                    reader.seek(max(0, log.stat().st_size - 16000))
                    tail = reader.read().decode(errors="replace")
                iterations = re.findall(r"Learning iteration\s+(\d+)/(\d+)", tail)
                progress = '<progress style="width:100%"></progress>'
                if iterations:
                    current, total = map(int, iterations[-1])
                    progress = f'<progress value="{current + 1}" max="{total}" style="width:100%"></progress>'
                elapsed = int(time.monotonic() - started)
                status.update(
                    HTML(
                        f'<div style="padding:14px;border:1px solid #76b900;border-radius:12px">'
                        f"<b>{html.escape(label)}</b> · {elapsed // 60}m {elapsed % 60}s"
                        f"{progress}<small>First launch may compile kernels and download assets.</small></div>"
                    )
                )
                with contextlib.suppress(subprocess.TimeoutExpired):
                    process.wait(timeout=1)
        except BaseException:
            with contextlib.suppress(ProcessLookupError):
                os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            status.update(HTML(f"Stopped. Full log: <code>{html.escape(str(log))}</code>"))
            raise
    text = log.read_text(errors="replace")
    tail = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", text[-6000:])
    state = "✅" if process.returncode == 0 else "❌"
    status.update(
        HTML(
            f'<div style="padding:14px;border:1px solid #aaa;border-radius:12px">'
            f"<b>{state} {html.escape(label)}</b> · {int(time.monotonic() - started)}s"
            f'<details><summary>Show log</summary><pre style="white-space:pre-wrap">'
            f"{html.escape(tail)}</pre></details><small>Full log: {html.escape(str(log))}</small></div>"
        )
    )
    if check and process.returncode:
        raise RuntimeError(f"{label} failed. Expand Show log above. Full log: {log}")
    return subprocess.CompletedProcess(command, process.returncode)


def video_snapshot():
    """Include modification times so rerunning a cell refreshes overwritten clips."""
    return {p: p.stat().st_mtime_ns for p in ROOT.rglob("*.mp4")}


def show_video(before, title="Your rollout"):
    videos = [p for p in ROOT.rglob("*.mp4") if before.get(p) != p.stat().st_mtime_ns]
    if not videos:
        raise RuntimeError("No fresh video was recorded. Expand the playback log above.")
    path = max(videos, key=lambda p: p.stat().st_mtime_ns)
    display(HTML(f"<h3>{html.escape(title)}</h3>"))
    display(Video(str(path), embed=True, width=640, html_attributes="controls loop muted playsinline"))
    return path


ISAACLAB_REF = "develop"  # @param {type:"string"}
ROOT = Path("/content/IsaacLab-3")
VIDEO_STEPS = 240  # @param {type:"slider", min:60, max:600, step:60}
RUN_ENV = os.environ | {"PYTHONUNBUFFERED": "1"}
RUN_ENV.pop("DISPLAY", None)

gpu = (
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if __import__("shutil").which("nvidia-smi")
    else None
)
if gpu is None or gpu.returncode:
    raise RuntimeError("Choose Runtime → Change runtime type → GPU, then rerun this cell.")
display(HTML(f"<p><b>GPU ready</b> · {html.escape(gpu.stdout.strip())}</p>"))
run([sys.executable, "-m", "pip", "install", "-q", "uv"], label="Installing uv")
if not ROOT.exists():
    run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            ISAACLAB_REF,
            "https://github.com/isaac-sim/IsaacLab.git",
            str(ROOT),
        ],
        label="Downloading Isaac Lab",
    )
else:
    branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=ROOT, text=True).strip()
    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
    requested = subprocess.run(["git", "rev-parse", ISAACLAB_REF], cwd=ROOT, capture_output=True, text=True)
    if branch != ISAACLAB_REF and (requested.returncode or requested.stdout.strip() != current_commit):
        raise RuntimeError(f"{ROOT} contains {branch}; set ISAACLAB_REF to that branch or use a new ROOT.")
run(
    ["uv", "sync", "--inexact", "--extra", "video", "--extra", "ov"],
    cwd=ROOT,
    env=RUN_ENV,
    label="Installing simulation dependencies",
)
revision = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
display(HTML(f"<p>✅ Ready · Isaac Lab revision <code>{revision}</code></p>"))

In [ ]:
# @title Set up the demo player
def isaaclab(*args, check=True):
    return run(
        ["uv", "run", "--no-sync", "isaaclab", *args],
        cwd=ROOT,
        env=RUN_ENV,
        check=check,
        label=f"Isaac Lab · {args[0]}",
    )


def play(task, *, physics, renderer=None, presets=None, checkpoint="pretrained"):
    before = video_snapshot()
    args = [
        "play",
        "--rl_library",
        "rsl_rl",
        "--task",
        task,
        "--checkpoint",
        checkpoint,
        "--num_envs",
        "1",
        "--seed",
        "42",
        "--video",
        "--video_length",
        str(VIDEO_STEPS),
        "--viz",
        "none" if renderer else "newton_gl",
        f"physics={physics}",
    ]
    if renderer:
        args += [f"renderer={renderer}", "--external_callback", "colab_camera.configure"]
    if presets:
        args.append(f"presets={presets}")
    result = isaaclab(*args, check=False)
    if result.returncode:
        display(
            HTML(
                "<p>Demo stopped. Expand <b>Show log</b> above for the cause. "
                "An unavailable checkpoint can be retried later; other demo cards are independent.</p>"
            )
        )
        return None
    return show_video(before, f"{task.removeprefix('Isaac-')} · {renderer or physics}")

In [ ]:
# @title Prepare the cloth demo
def initialize_and_play(task: str, *, physics: str) -> None:
    """Create a local one-iteration checkpoint, then record its untrained behavior."""
    train_result = isaaclab(
        "train",
        "--rl_library",
        "rsl_rl",
        "--task",
        task,
        "--num_envs",
        "1",
        "--max_iterations",
        "1",
        "--run_name",
        "colab_physics_demo",
        f"physics={physics}",
        check=False,
    )
    if train_result.returncode:
        print("⚠️ The local initialization run failed; read its output above.")
        return
    before = video_snapshot()
    play_result = isaaclab(
        "play",
        "--rl_library",
        "rsl_rl",
        "--task",
        task,
        "--checkpoint",
        "latest",
        "--num_envs",
        "1",
        "--video",
        "--video_length",
        str(VIDEO_STEPS),
        "--viz",
        "newton_gl",
        f"physics={physics}",
        check=False,
    )
    if play_result.returncode:
        print("⚠️ Playback failed; read its output above.")
        return
    show_video(before, "Cloth · untrained physics demo")

## 2 · Meet your robot 🐾
ANYmal-D follows velocity commands using its body and joint state. Run once, then change the physics selector and compare the gait.

**Watch:** foot placement, balance, and turns. Each backend loads its matching trained policy.


In [ ]:
# @title Choose physics and press play
physics = "newton_mjwarp"  # @param ["newton_mjwarp", "ovphysx"]
play("Isaac-Velocity-Flat-AnymalD", physics=physics)

## 3 · Pick the next act 🦾
A humanoid on rough ground or a robot arm lifting a cube?

**Watch:** G1's balance corrections, or Franka's approach → grasp → lift.


In [ ]:
# @title Pick a robot
robot = "Franka · lift a cube"  # @param ["Franka · lift a cube", "G1 · rough terrain"]
task = {"Franka · lift a cube": "Isaac-Lift-Franka", "G1 · rough terrain": "Isaac-Velocity-Rough-G1"}[robot]
play(task, physics="newton_mjwarp")

## 4 · See through the policy's camera 📷
These clips show the **RGB sensor input** used by the policy. Keep physics fixed and switch renderers.

**Watch:** edges, lighting, and materials. Each renderer loads a matching checkpoint; these are separate rollouts.


In [ ]:
# @title Prepare sensor recording
from textwrap import dedent

# Keep the original task ID so published-checkpoint lookup still matches.
camera_module = dedent("""
    import gymnasium as gym
    from isaaclab.envs.utils.video_recorder_cfg import VideoRecorderCfg
    from isaaclab_tasks.core.cartpole.cartpole_manager_camera_env_cfg import CartpoleCameraEnvCfg

    def camera_cfg():
        cfg = CartpoleCameraEnvCfg()
        cfg.rgb.video_recorders = [VideoRecorderCfg(source="sensor:tiled_camera:rgb")]
        cfg.default = cfg.rgb
        return cfg

    def configure():
        gym.spec("Isaac-Cartpole-Camera").kwargs["env_cfg_entry_point"] = camera_cfg
""")
(ROOT / "colab_camera.py").write_text(camera_module)
RUN_ENV["PYTHONPATH"] = str(ROOT) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")
print("✅ Camera recorder ready.")

In [ ]:
# @title Choose a renderer and press play
renderer = "newton_renderer"  # @param ["newton_renderer", "ovrtx"]
play("Isaac-Cartpole-Camera", physics="newton_mjwarp", renderer=renderer, presets="rgb")

## 5 · Let it bend 🪢
MJWarp moves the rigid robot; VBD simulates the cloth. This card creates a tiny local training run, then records it.

**Watch:** the cloth deform. This is an **untrained physics demo**, so don't expect a successful lift.


In [ ]:
# @title Run the cloth demo
initialize_and_play("Isaac-Lift-Cloth-Franka", physics="newton_mjwarp_vbd_proxy")

## Your turn
Try both physics backends and both renderers, then [train your own walking policy](https://colab.research.google.com/github/isaac-sim/IsaacLab/blob/develop/notebooks/training.ipynb).

<details><summary>Stuck?</summary>

- **No GPU:** select a GPU runtime and rerun setup.
- **Out of memory:** stop the running cell; restart the runtime if memory remains occupied.
- **Missing checkpoint or asset:** expand the log, check connectivity, and retry.
- **Graphics/driver error:** try another Colab GPU or a compatible local runtime.
- **Keep a clip:** download its MP4 from Colab's Files pane under `IsaacLab-3`.

</details>
